In [48]:
from kanren import *

# Task 1 - Family Tree Puzzle

Sample Dataset: Windsor Family tree

In [49]:
'''
base relations
'''
parent_of = Relation() # parent_of(parent, child)
male = Relation()
female = Relation()

gender_facts = [
    fact(male, 'George_VI'),
    fact(male, 'Philip'),
    fact(male, 'Charles'),
    fact(male, 'Andrew'),
    fact(male, 'Edward'),
    fact(male, 'William'),
    fact(male, 'Harry'),
    fact(male, 'Peter_Phillips'),
    fact(male, 'George'),
    fact(male, 'Louis'),
    fact(male, 'Archie'),
    fact(male, 'James'),

    fact(female, 'Elizabeth_BL'), # quen mother (bowes lyon)
    fact(female, 'Elizabeth_II'), # 2nd
    fact(female, 'Margaret'),
    fact(female, 'Anne'),
    fact(female, 'Diana'),
    fact(female, 'Catherine'),
    fact(female, 'Meghan'),
    fact(female, 'Zara'),
    fact(female, 'Beatrice'),
    fact(female, 'Eugenie'),
    fact(female, 'Louise'),
    fact(female, 'Charlotte'),
    fact(female, 'Lilibet'),
]

# parentOf facts - (parent, child) format
parent_facts = [
    fact(parent_of, 'George_VI', 'Elizabeth_II'),
    fact(parent_of, 'George_VI', 'Margaret'),
    fact(parent_of, 'Elizabeth_BL', 'Elizabeth_II'),
    fact(parent_of, 'Elizabeth_BL', 'Margaret'),

    fact(parent_of, 'Elizabeth_II', 'Charles'),
    fact(parent_of, 'Elizabeth_II', 'Anne'),
    fact(parent_of, 'Elizabeth_II', 'Andrew'),
    fact(parent_of, 'Elizabeth_II', 'Edward'),
    fact(parent_of, 'Philip', 'Charles'),
    fact(parent_of, 'Philip', 'Anne'),
    fact(parent_of, 'Philip', 'Andrew'),
    fact(parent_of, 'Philip', 'Edward'),

    fact(parent_of, 'Charles', 'William'),
    fact(parent_of, 'Charles', 'Harry'),
    fact(parent_of, 'Diana', 'William'),
    fact(parent_of, 'Diana', 'Harry'),

    fact(parent_of, 'Anne', 'Peter_Phillips'),
    fact(parent_of, 'Anne', 'Zara'),

    fact(parent_of, 'Andrew', 'Beatrice'),
    fact(parent_of, 'Andrew', 'Eugenie'),

    fact(parent_of, 'Edward','Louise'),
    fact(parent_of, 'Edward','James'),

    fact(parent_of, 'William', 'George'),
    fact(parent_of, 'William', 'Charlotte'),
    fact(parent_of, 'William', 'Louis'),
    fact(parent_of, 'Catherine', 'George'),
    fact(parent_of, 'Catherine', 'Charlotte'),
    fact(parent_of, 'Catherine', 'Louis'),

    fact(parent_of, 'Harry', 'Archie'),
    fact(parent_of, 'Harry', 'Lilibet'),
    fact(parent_of, 'Meghan', 'Archie'),
    fact(parent_of, 'Meghan', 'Lilibet'),
]

In [50]:
def are_siblings(x, y):
    """
    - share at least one common parent
    - x!=y
    """
    p = var()
    return lall(
        neq(x, y),
        parent_of(p, x),
        parent_of(p, y)
    )

def are_sisters(x, y):
    """
    - are siblings
    - both female
    """
    return lall(
        are_siblings(x, y),
        female(x),
        female(y)
    )


def are_brothers(x, y):
    """
    - are siblings
    - both male
    """
    return lall(
        are_siblings(x, y),
        male(x),
        male(y)
    )


def is_mother_of(x, y): # (mother, child)
    """
    x is mother if:
    - x is parent of y
    - x is female
    """
    return lall(
        parent_of(x, y),
        female(x)
    )


def is_father_of(x, y): # (father, child)
    """
    x is father of y if:
    - x is parent of y
    - x is male
    """
    return lall(
        parent_of(x, y),
        male(x)
    )

def is_grandmother_of(x, y):
    """
    x is grandma of y if:
    - x is mother of some person p
    - - such that p is a parent of y
    """
    p = var()
    return lall(
        is_mother_of(x,p),
        parent_of(p,y)
    )

def is_grandfather_of(x, y):
    """
    x is grandfather of y if:
    - x is father of some person p
    - - such that p is a parent of y
    """
    p = var()
    return lall(
        is_father_of(x, p),
        parent_of(p, y)
    )

def are_cousins(x, y):
    """
    x and y are 1st cousins if:
    - parents are siblings
    - x!=y
    """
    parent_of_x = var()
    parent_of_y = var()
    return lall(
        parent_of(parent_of_x, x),
        parent_of(parent_of_y, y),
        are_siblings(parent_of_x, parent_of_y),
        neq(x,y)
    )

In [51]:
kb = lall(
    *gender_facts,
    *parent_facts
)

### Sample Queries

In [52]:
# 1. Are William and Harry cousins?
# expected: no - they are brothers, not cousins
q = var()
solution = run(0, q, are_cousins('William', 'Harry'))
print('Are William and Harry cousins?', 'Yes' if solution else 'No')

# 2. Are Beatrice and Eugenie sisters?
# expected: yes - both daughters of Andrew
solution = run(0, q, are_sisters('Beatrice', 'Eugenie'))
print('Are Beatrice and Eugenie sisters?', 'Yes' if solution else 'No')

# 3. Who is the grandmother of George?
# expected: Diana
gm = var('grandmother')
solution = run(0, gm, is_grandmother_of(gm, 'George'))
print('Grandmother of George:', solution)

# 4. Are George and Archie cousins?
# expected: yes - george's parent - william; archie's parent - harry; william and harry - siblings
solution = run(0, q, are_cousins('George', 'Archie'))
print('Are George and Archie cousins?', 'Yes' if solution else 'No')

# 5. list all cousins of charlotte
cousin = var('cousin')
solution = run(0, cousin, are_cousins(cousin, 'Charlotte'))
print('Cousins of Charlotte:', set(solution)) # using set to remove duplicates

Are William and Harry cousins? No
Are Beatrice and Eugenie sisters? Yes
Grandmother of George: ('Diana',)
Are George and Archie cousins? Yes
Cousins of Charlotte: {'Archie', 'Lilibet'}


# Task 2 - knights and knaves puzzle

**Description of the puzzle (in English):**

Premise:
- On an island, knights always tell the truth, and knaves always lie.
- You meet three inhabitants: John, David and Michael.

You have the following information:
- John says: "David is a knave."
- David says: "I know that John is a knight and that Michael is a knave."
- Michael says: "I and David are different."

Questions:
- Who is knave?
- Who is knight?

## Solution: express the facts, rules, and query

In [53]:
john, david, michael = vars(3)

### Knowledge base for knights and knaves

core principle:
- knight's statement is `true`
- knave's statement is `false`

each statement will be encoded as a constraint on the variables

#### Statement encoding:
- john says "david is knave"; if john is:
    - `knight` -> statement is *true* -> david is `knave`
    - `knave` -> statement is *false* -> david is `knight`

- david says "john is knight *and* bob is knave"; if david is:
    - `knight` -> statement is *true* -> john=`knight` and bob=`knave`
    - `knave` -> statement is *false* (truth is inverse of that) -> NOT(john=`knigh` and bob=`knave`) -> john=`knave` or bob=`knight`

- bob says "me and david are different" (one knight, one knave); if bob is:
    - `knight` -> statement is *true* -> bob != david
    - `knave` -> statement is *false* -> bob = david

In [54]:
kb = lall(
    membero(john, ['knight', 'knave']),
    membero(david, ['knight', 'knave']),
    membero(michael, ['knight', 'knave']),

    # john: "david is knave"
    conde(
        # case 1: john is knight -> statement is true -> david is knave
        [eq(john, 'knight'), eq(david, 'knave')],
        # case 2: john is knave -> statement is false -> david is knight
        [eq(john, 'knave'),  eq(david, 'knight')]
    ),

    # david: "john is knight and mike is knave"
    conde(
        # case 1: david is knight -> statement is true
        [eq(david, 'knight'), eq(john, 'knight'), eq(michael, 'knave')],
        # case 2: david is knave -> statement is false
        # john='knave' or michael='knight'
        [eq(david, 'knave'), conde(
            [eq(john, 'knave')], # subcase 2.1: john is knave
            [eq(michael, 'knight')] # subcase 2.2: michael is knight
        )]
    ),

    # micheal: "me and david are different"
    conde(
        # case 1: micheal is knight -> statement is true -> michael != david
        [eq(michael, 'knight'), neq(michael, david)],
        # case 2: michael is knave -> statement is false -> michael = david
        [eq(michael, 'knave'),  eq(michael, david)]
    )
)

In [55]:
solutions = run(0, (john, david, michael), kb)

# deduplicate
solutions = list(dict.fromkeys(solutions))

print(f'number of solutions found: {len(solutions)}\n')

john_role, david_role, michael_role = solutions[0]
print('John is a', john_role)
print('David is a', david_role)
print('Michael is a', michael_role)

number of solutions found: 1

John is a knight
David is a knave
Michael is a knight
